# Load from raw CSV

In [1]:
from pathlib import Path
import pandas as pd

# # Tìm file CSV CSI mới nhất trong thư mục hiện tại
# csv_files = sorted(Path('.').glob('csi_data_*.csv'), key=lambda p: p.stat().st_mtime)

# if not csv_files:
#     raise FileNotFoundError("Không tìm thấy file dạng csi_data_*.csv trong thư mục hiện tại")

# latest_csv = csv_files[-1]

latest_csv = 'Router/khong nguoi/csi_data_20260305_143656.csv'
print(f"Dang doc: {latest_csv}")

df = pd.read_csv(latest_csv)

print("\nKích thước dữ liệu:", df.shape)
print("\nTên cột:")
print(df.columns.tolist())

print("\n5 dòng đầu:")
display(df.head())

Dang doc: Router/khong nguoi/csi_data_20260305_143656.csv

Kích thước dữ liệu: (4000, 15)

Tên cột:
['type', 'id', 'mac', 'rssi', 'rate', 'noise_floor', 'fft_gain', 'agc_gain', 'channel', 'local_timestamp', 'sig_len', 'rx_state', 'len', 'first_word', 'data']

5 dòng đầu:


,type,id,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
0,CSI_DATA,20100,98:4a:6b:31:4a:10,-57,11,-99,-51,57,11,207653569,83,0,256,0,"[0,0,0,0,0,0,0,0,-6,-6,-5,-7,-4,-9,-1,-10,0,-1..."
1,CSI_DATA,20101,98:4a:6b:31:4a:10,-58,11,-99,-48,57,11,207661711,83,0,256,0,"[0,0,0,0,0,0,0,0,-8,-1,-8,-3,-8,-5,-7,-6,-6,-8..."
2,CSI_DATA,20102,98:4a:6b:31:4a:10,-57,11,-99,-48,57,11,207670030,83,0,256,0,"[0,0,0,0,0,0,0,0,-2,-9,0,-9,1,-9,3,-9,5,-9,7,-..."
3,CSI_DATA,20103,98:4a:6b:31:4a:10,-57,11,-99,-47,57,11,207678083,83,0,256,0,"[0,0,0,0,0,0,0,0,0,9,-2,8,-3,8,-5,8,-7,6,-9,5,..."
4,CSI_DATA,20104,98:4a:6b:31:4a:10,-57,11,-99,-48,56,11,207686405,83,0,256,0,"[0,0,0,0,0,0,0,0,-7,-5,-7,-6,-5,-7,-4,-9,-3,-1..."


# Calculate Amplitude & phase

In [2]:
import json
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import os
import time

# Parse chuỗi JSON trong cột `data` -> mảng số
def parse_csi_data(value):
    if pd.isna(value):
        return np.array([], dtype=np.float64)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float64)
    try:
        return np.asarray(json.loads(value), dtype=np.float64)
    except Exception:
        return np.array([], dtype=np.float64)

# Dữ liệu CSI format C5/C6: [imag0, real0, imag1, real1, ...]
def compute_amplitude_phase(csi_raw):
    if csi_raw.size < 2:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)

    imag = csi_raw[0::2]
    real = csi_raw[1::2]

    length = min(len(real), len(imag))
    real = real[:length]
    imag = imag[:length]

    amplitude = np.sqrt(real**2 + imag**2)
    phase = np.arctan2(imag, real)
    return amplitude, phase

def process_packet(data_raw):
    """Parse CSI data và tính amplitude/phase cho 1 packet."""
    csi_array = parse_csi_data(data_raw)
    amp, phs = compute_amplitude_phase(csi_array)
    return csi_array, amp, phs

# Tính cho toàn bộ dataframe với parallel processing
df_calc = df.copy()

print(f"Tính Amplitude & Phase với thread-based parallel processing...")
print(f"Tổng {len(df_calc)} packets\n")

num_workers = max(1, os.cpu_count() - 1)
print(f"Dùng {num_workers} threads...\n")

start_time = time.time()

# Parallel processing với ThreadPoolExecutor (tránh pickle issues)
csi_raw_list = []
amplitude_list = []
phase_list = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(process_packet, raw) for raw in df_calc["data"]]
    
    completed = 0
    for future in futures:
        try:
            csi_array, amp, phs = future.result()
            csi_raw_list.append(csi_array)
            amplitude_list.append(amp)
            phase_list.append(phs)
            completed += 1
            if completed % max(1, len(df_calc)//10) == 0:
                print(f"  Đã xử lý {completed}/{len(df_calc)} packets...")
        except Exception as e:
            print(f"  Lỗi xử lý packet: {e}")
            csi_raw_list.append(np.array([]))
            amplitude_list.append(np.array([]))
            phase_list.append(np.array([]))
            completed += 1

elapsed = time.time() - start_time

df_calc["csi_raw"] = csi_raw_list
df_calc["amplitude"] = amplitude_list
df_calc["phase"] = phase_list

print(f"Hoàn tất tính amplitude & phase (took {elapsed:.2f}s)\n")

# Feature thống kê nhanh theo từng packet
df_calc["subcarrier_count"] = df_calc["amplitude"].apply(len)
df_calc["amp_mean"] = df_calc["amplitude"].apply(lambda x: float(np.mean(x)) if len(x) else np.nan)
df_calc["amp_std"] = df_calc["amplitude"].apply(lambda x: float(np.std(x)) if len(x) else np.nan)
df_calc["phase_mean"] = df_calc["phase"].apply(lambda x: float(np.mean(x)) if len(x) else np.nan)
df_calc["phase_std"] = df_calc["phase"].apply(lambda x: float(np.std(x)) if len(x) else np.nan)

print("Hoàn tất tính feature thống kê")
print("Số packet:", len(df_calc))
print("Subcarrier/packet (min/max):", int(df_calc["subcarrier_count"].min()), "/", int(df_calc["subcarrier_count"].max()))

display(
    df_calc[["rssi", "len", "subcarrier_count", "amp_mean", "amp_std", "phase_mean", "phase_std"]].head()
 )

# Xem chi tiết packet đầu tiên
first_idx = df_calc["subcarrier_count"].gt(0).idxmax()
print(f"\nPacket mẫu index: {first_idx}")
print("5 amplitude đầu:", df_calc.loc[first_idx, "amplitude"][:5])
print("5 phase đầu:", df_calc.loc[first_idx, "phase"][:5])

Tính Amplitude & Phase với thread-based parallel processing...
Tổng 4000 packets

Dùng 7 threads...

  Đã xử lý 400/4000 packets...
  Đã xử lý 800/4000 packets...
  Đã xử lý 1200/4000 packets...
  Đã xử lý 1600/4000 packets...
  Đã xử lý 2000/4000 packets...
  Đã xử lý 2400/4000 packets...
  Đã xử lý 2800/4000 packets...
  Đã xử lý 3200/4000 packets...
  Đã xử lý 3600/4000 packets...
  Đã xử lý 4000/4000 packets...
Hoàn tất tính amplitude & phase (took 0.20s)

Hoàn tất tính feature thống kê
Số packet: 4000
Subcarrier/packet (min/max): 128 / 128


,rssi,len,subcarrier_count,amp_mean,amp_std,phase_mean,phase_std
0,-57,256,128,9.115841,4.142373,0.160937,1.777651
1,-58,256,128,8.726680,4.030321,-0.090851,1.766804
2,-57,256,128,9.171939,4.191758,0.249588,1.769468
3,-57,256,128,9.211060,4.184248,-0.059697,1.692818
4,-57,256,128,9.468200,4.310423,0.059219,1.792746



Packet mẫu index: 0
5 amplitude đầu: [0.         0.         0.         0.         8.48528137]
5 phase đầu: [ 0.          0.          0.          0.         -2.35619449]


## Unwarp phase

In [3]:
def unwrap_phase(phase_list):
    """Thực hiện unwrap pha cho từng packet."""
    return [np.unwrap(p) if len(p) > 0 else p for p in phase_list]

# Áp dụng cho dữ liệu
print("Đang thực hiện Phase Unwrapping...")
df_calc["phase_unwrapped"] = unwrap_phase(df_calc["phase"])

Đang thực hiện Phase Unwrapping...


## Phase Sanitization

In [4]:
def sanitize_phase_robust(phase_array):
    """
    Loại bỏ lỗi pha tuyến tính (Linear Phase Offset) do SFO/CFO.
    Sử dụng Linear Regression trên các subcarrier có giá trị.
    """
    if len(phase_array) == 0:
        return phase_array
    
    # Chỉ lấy các subcarrier có pha khác 0 (tránh guard subcarriers / DC)
    valid_idx = np.where(np.abs(phase_array) > 1e-6)[0]
    if len(valid_idx) < 2:
        return phase_array
        
    x = valid_idx
    y = phase_array[valid_idx]
    
    # Linear fit: y = ax + b
    a, b = np.polyfit(x, y, 1)
    
    sanitized = phase_array.copy()
    sanitized[valid_idx] = y - (a * x + b)
    return sanitized

print("Đang thực hiện Phase Sanitization...")
df_calc["phase_sanitized"] = df_calc["phase_unwrapped"].apply(sanitize_phase_robust)

Đang thực hiện Phase Sanitization...


## Display sample subcarrier

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

SUBCARRIER_INDEX=10

# Chọn packet đầu tiên có dữ liệu hợp lệ
valid_packets = df_calc[df_calc["subcarrier_count"] > 0]
if valid_packets.empty:
    raise ValueError("Không có packet hợp lệ để plot")

sample_idx = valid_packets.index[0]
amp = np.asarray(df_calc.loc[sample_idx, "amplitude"])
phs = np.asarray(df_calc.loc[sample_idx, "phase"])
subcarrier_idx = np.arange(len(amp))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[f"Amplitude - Packet {sample_idx}",
                                    f"Phase - Packet {sample_idx}"])

fig.add_trace(go.Scatter(x=subcarrier_idx, y=amp, mode='lines',
                          line=dict(color='royalblue', width=1.2),
                          name='Amplitude'), row=1, col=1)

fig.add_trace(go.Scatter(x=subcarrier_idx, y=phs, mode='lines',
                          line=dict(color='darkorange', width=1.2),
                          name='Phase'), row=2, col=1)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Phase (rad)", row=2, col=1)
fig.update_xaxes(title_text="Subcarrier Index", row=2, col=1)
fig.update_layout(height=700, width=1200, showlegend=True,
                  template='plotly_white')
fig.show()

# Extract Subcarrier 10 Time Series

## Remove guard subcarriers

In [6]:
import numpy as np
import pandas as pd

# 1. Chuyển toàn bộ cột thành ma trận 2D (Số packet x Số subcarriers)
# Giả sử mỗi hàng trong df_calc['amplitude'] có độ dài bằng nhau
amp_matrix = np.stack(df_calc["amplitude"].values)
phs_matrix = np.stack(df_calc["phase_sanitized"].values)

# 2. Định nghĩa vị trí cắt (Bỏ guard subcarriers)
GUARD_FRONT = 5
GUARD_BACK = 4
data_start = GUARD_FRONT
data_end = amp_matrix.shape[1] - GUARD_BACK

# 3. Trích xuất bằng Slicing (Cực nhanh)
amp_data = amp_matrix[:, data_start:data_end]
phs_data = phs_matrix[:, data_start:data_end]

# 4. Tổ chức lại vào dictionary (nếu bạn vẫn muốn dùng cấu trúc cũ)
ts_all = {}
packet_indices = df_calc.index.values

for i, sub_idx in enumerate(range(data_start, data_end)):
    # Tạo DataFrame một lần duy nhất cho mỗi subcarrier từ ma trận đã cắt
    ts_all[sub_idx] = pd.DataFrame({
        "packet_idx": packet_indices,
        "amp": amp_data[:, i],
        "phase": phs_data[:, i]
    })

print(f"Trích xuất thành công {len(ts_all)} subcarrier.")

Trích xuất thành công 119 subcarrier.


# Hampel Filter

In [7]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import os
import time

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False

    def njit(*args, **kwargs):
        if args and callable(args[0]) and len(args) == 1 and not kwargs:
            return args[0]

        def decorator(func):
            return func

        return decorator


K_WINDOW_AMP = 50 # 200 => 30
N_SIGMA_AMP = 3.0

K_WINDOW_PHS = 30 # 200 => 30
N_SIGMA_PHS = 2.0


@njit(cache=True, fastmath=True)
def hampel_filter_numba(data, window_size, n_sigma):
    n = len(data)
    filtered = data.copy()
    outliers = np.zeros(n, dtype=np.bool_)
    half_window = window_size // 2

    for i in range(n):
        start = i - half_window
        if start < 0:
            start = 0

        end = i + half_window + 1
        if end > n:
            end = n

        window = data[start:end]
        if window.size == 0:
            continue

        median_val = np.median(window)
        mad = np.median(np.abs(window - median_val))

        if mad > 0.0 and np.abs(data[i] - median_val) > (n_sigma * mad):
            filtered[i] = median_val
            outliers[i] = True

    return filtered, outliers


def _reflect_pad(values, desired_pad):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return values, 0

    pad_len = min(int(desired_pad), max(0, values.size - 1))
    if pad_len <= 0:
        return values.copy(), 0

    return np.pad(values, (pad_len, pad_len), mode='reflect'), pad_len


def apply_hampel_single_numba(sub_idx, ts_df, k_window_amp, n_sigma_amp, k_window_phs, n_sigma_phs):
    """Hampel filter cho một subcarrier bằng Numba."""
    if ts_df.empty:
        return sub_idx, {
            "amp_hampel": np.array([], dtype=np.float64),
            "amp_hampel_outliers": np.array([], dtype=bool),
            "phase_hampel": np.array([], dtype=np.float64),
            "phase_hampel_outliers": np.array([], dtype=bool),
        }

    pad_len = max(k_window_amp, k_window_phs) * 2
    amp_padded, amp_pad_len = _reflect_pad(ts_df["amp"].values, pad_len)
    phs_padded, phs_pad_len = _reflect_pad(ts_df["phase"].values, pad_len)

    amp_hampel_padded, amp_hampel_outliers_padded = hampel_filter_numba(
        amp_padded,
        k_window_amp,
        n_sigma_amp,
    )
    phs_hampel_padded, phs_hampel_outliers_padded = hampel_filter_numba(
        phs_padded,
        k_window_phs,
        n_sigma_phs,
    )

    amp_hampel = amp_hampel_padded[amp_pad_len:-amp_pad_len] if amp_pad_len > 0 else amp_hampel_padded
    amp_hampel_outliers = amp_hampel_outliers_padded[amp_pad_len:-amp_pad_len].copy() if amp_pad_len > 0 else amp_hampel_outliers_padded.copy()
    phs_hampel = phs_hampel_padded[phs_pad_len:-phs_pad_len] if phs_pad_len > 0 else phs_hampel_padded
    phs_hampel_outliers = phs_hampel_outliers_padded[phs_pad_len:-phs_pad_len].copy() if phs_pad_len > 0 else phs_hampel_outliers_padded.copy()

    return sub_idx, {
        "amp_hampel": amp_hampel,
        "amp_hampel_outliers": amp_hampel_outliers,
        "phase_hampel": phs_hampel,
        "phase_hampel_outliers": phs_hampel_outliers,
    }


print(f"Áp dụng Hampel filter bằng Numba (available={NUMBA_AVAILABLE}) với k={K_WINDOW_AMP}/{K_WINDOW_PHS} cho tất cả {len(ts_all)} subcarrier...\n")

if NUMBA_AVAILABLE:
    _warmup = np.array([1.0, 2.0, 3.0, 100.0, 4.0, 5.0, 6.0], dtype=np.float64)
    _warmup_padded, _ = _reflect_pad(_warmup, 6)
    _ = hampel_filter_numba(_warmup_padded, 3, 3.0)

num_workers = min(len(ts_all), max(1, os.cpu_count() - 1))
print(f"Dùng {num_workers} threads cho {len(ts_all)} subcarriers...\n")

start_time = time.time()

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [
        executor.submit(
            apply_hampel_single_numba,
            sub_idx,
            ts_all[sub_idx],
            K_WINDOW_AMP,
            N_SIGMA_AMP,
            K_WINDOW_PHS,
            N_SIGMA_PHS,
        )
        for sub_idx in ts_all.keys()
    ]

    completed = 0
    for future in futures:
        try:
            sub_idx, result_dict = future.result()
            for key, val in result_dict.items():
                ts_all[sub_idx][key] = val
            completed += 1
            if completed % max(1, len(ts_all) // 10) == 0:
                print(f"  Đã xử lý {completed}/{len(ts_all)} subcarriers...")
        except Exception as e:
            print(f"  Lỗi xử lý subcarrier: {e}")
            completed += 1

elapsed = time.time() - start_time
print(f"Hoàn tất Hampel filter cho tất cả subcarrier (took {elapsed:.2f}s)")

# Hiển thị thống kê cho subcarrier 10
ts_sample = ts_all[SUBCARRIER_INDEX]
print(f"\nThống kê sau Hampel - Subcarrier {SUBCARRIER_INDEX}:")


Áp dụng Hampel filter bằng Numba (available=True) với k=50/30 cho tất cả 119 subcarrier...

Dùng 7 threads cho 119 subcarriers...

  Đã xử lý 11/119 subcarriers...
  Đã xử lý 22/119 subcarriers...
  Đã xử lý 33/119 subcarriers...
  Đã xử lý 44/119 subcarriers...
  Đã xử lý 55/119 subcarriers...
  Đã xử lý 66/119 subcarriers...
  Đã xử lý 77/119 subcarriers...
  Đã xử lý 88/119 subcarriers...
  Đã xử lý 99/119 subcarriers...
  Đã xử lý 110/119 subcarriers...
Hoàn tất Hampel filter cho tất cả subcarrier (took 1.33s)

Thống kê sau Hampel - Subcarrier 10:


## Display Sample 

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Chỉ plot subcarrier 10
ts_df = ts_all[SUBCARRIER_INDEX]
packet_idx = np.arange(len(ts_df))

# Lấy ra index của outlier
amp_outlier_idx = np.where(ts_df['amp_hampel_outliers'])[0]
phs_outlier_idx = np.where(ts_df['phase_hampel_outliers'])[0]

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=[
                        f"Amplitude - Subcarrier {SUBCARRIER_INDEX} (Before Hampel)"
                    ])

# Plot 1: Amplitude - Trước Hampel
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp'], mode='lines',
                          line=dict(color='royalblue', width=1.2),
                          opacity=0.7, name="Original"),
              row=1, col=1)

# Plot 2: Amplitude - Sau Hampel
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_hampel'], mode='lines',
                          line=dict(color='green', width=1.2),
                          opacity=0.7, name="Hampel Filtered"),
              row=2, col=1)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Amplitude", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_layout(height=500, width=1200, showlegend=True,
                  template='plotly_white')
fig.show()

# Savitzky-Golay Filter (Smoothing)

In [9]:
from scipy.signal import savgol_filter
import numpy as np

# Tham số Savitzky-Golay amp
POLYORDER_AMP = 3      # Bậc đa thức
WINDOW_LENGTH_AMP = 31 # Kích thước cửa sổ

# Tham số Savitzky-Golay phase
POLYORDER_PHASE = 3    # Bậc đa thức
WINDOW_LENGTH_PHASE = 41 # Kích thước cửa sổ

print(f"Áp dụng Savitzky-Golay Filter (p={POLYORDER_AMP}, l={WINDOW_LENGTH_AMP}) cho tất cả subcarriers...\n")

# Áp dụng SG filter cho tất cả subcarriers
for sub_idx in ts_all.keys():
    ts_df = ts_all[sub_idx]
    
    amp_len = len(ts_df)
    window_len = min(WINDOW_LENGTH_AMP, amp_len if amp_len % 2 == 1 else amp_len - 1)
    
    # Áp dụng SG filter từ amplitude & phase đã qua Hampel
    if window_len >= POLYORDER_AMP + 1:
        amp_sg = savgol_filter(ts_df['amp_hampel'].values, window_length=window_len, polyorder=POLYORDER_AMP)
    else:
        amp_sg = ts_df['amp_hampel'].values
    
    # Lưu vào dataframe
    ts_all[sub_idx]["amp_sg"] = amp_sg

for sub_idx in ts_all.keys():
    ts_df = ts_all[sub_idx]
    
    window_len = min(WINDOW_LENGTH_PHASE, amp_len if amp_len % 2 == 1 else amp_len - 1)
    
    if window_len >= POLYORDER_PHASE + 1:
        phs_sg = savgol_filter(ts_df['phase_hampel'].values, window_length=window_len, polyorder=POLYORDER_PHASE)
    else:
        phs_sg = ts_df['phase_hampel'].values
    
    # Lưu vào dataframe
    ts_all[sub_idx]["phase_sg"] = phs_sg

print(f"Hoàn tất SG filter cho tất cả {len(ts_all)} subcarriers")

# Hiển thị thống kê cho subcarrier tham chiếu
ts_sample = ts_all[SUBCARRIER_INDEX]
print(f"Thống kê cho Subcarrier {SUBCARRIER_INDEX} sau Savitzky-Golay:")
print(f"  Amplitude: mean={np.mean(ts_sample['amp_sg']):.3f}, std={np.std(ts_sample['amp_sg']):.3f}")
print(f"  Phase: mean={np.mean(ts_sample['phase_sg']):.3f}, std={np.std(ts_sample['phase_sg']):.3f}")

# So sánh MSE giữa Hampel và SG
amp_mse = np.mean((ts_sample['amp_hampel'].values - ts_sample['amp_sg'].values)**2)
phs_mse = np.mean((ts_sample['phase_hampel'].values - ts_sample['phase_sg'].values)**2)

print(f"\nMSE (Hampel → SG) cho Subcarrier {SUBCARRIER_INDEX}:")
print(f"  Amplitude: {amp_mse:.6f}")
print(f"  Phase: {phs_mse:.6f}")

Áp dụng Savitzky-Golay Filter (p=3, l=31) cho tất cả subcarriers...

Hoàn tất SG filter cho tất cả 119 subcarriers
Thống kê cho Subcarrier 10 sau Savitzky-Golay:
  Amplitude: mean=10.976, std=0.222
  Phase: mean=9.116, std=0.407

MSE (Hampel → SG) cho Subcarrier 10:
  Amplitude: 0.124326
  Phase: 0.514094


# Visualization - Hampel vs Savitzky-Golay vs Final

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Lấy dữ liệu từ dict cho subcarrier 10
ts_df = ts_all[SUBCARRIER_INDEX]

if ts_df.empty:
    raise ValueError("Time series dataframe trống, không thể vẽ")

packet_idx = np.arange(len(ts_df))

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[
                        f"Amplitude - Original (SC {SUBCARRIER_INDEX})",
                        f"Phase - Original (SC {SUBCARRIER_INDEX})",
                    ])

# Row 1: Amplitude
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp'], mode='lines',
                          line=dict(color='royalblue', width=1), opacity=0.7,
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_hampel'], mode='lines',
                          line=dict(color='green', width=1), opacity=0.7,
                          showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_sg'], mode='lines',
                          line=dict(color='purple', width=1), opacity=0.7,
                          showlegend=False), row=1, col=3)

# Row 2: Phase
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase'], mode='lines',
                          line=dict(color='darkorange', width=1), opacity=0.7,
                          showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase_hampel'], mode='lines',
                          line=dict(color='green', width=1), opacity=0.7,
                          showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase_sg'], mode='lines',
                          line=dict(color='purple', width=1), opacity=0.7,
                          showlegend=False), row=2, col=3)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Phase (rad)", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=2)
fig.update_xaxes(title_text="Packet Index", row=2, col=3)
fig.update_layout(height=600, width=1600, showlegend=False,
                  template='plotly_white')
fig.show()

# Elliptic Bandpass Filter (0.1–0.4 Hz)

In [11]:
import numpy as np
from scipy.signal import ellip, sosfiltfilt

# ============ Cấu hình chung ============
FS = 100.0  # Sampling frequency (Hz)
PADLEN = 3000 
nyquist = FS / 2

# ============ 1. Cấu hình Elliptic cho AMPlitude ============
LOWCUT_AMP = 0.15
HIGHCUT_AMP = 0.5
ORDER_AMP = 4
RP_AMP = 1
RS_AMP = 40

sos_amp = ellip(N=ORDER_AMP, rp=RP_AMP, rs=RS_AMP, 
                Wn=[LOWCUT_AMP/nyquist, HIGHCUT_AMP/nyquist], 
                btype='bandpass', output='sos')

# ============ 2. Cấu hình Elliptic cho PHASE ============
# Bạn có thể thử dải tần rộng hơn hoặc hẹp hơn cho Phase
LOWCUT_PHS = 0.1 
HIGHCUT_PHS = 0.6
ORDER_PHS = 2
RP_PHS = 0.5
RS_PHS = 50

sos_phs = ellip(N=ORDER_PHS, rp=RP_PHS, rs=RS_PHS, 
                Wn=[LOWCUT_PHS/nyquist, HIGHCUT_PHS/nyquist], 
                btype='bandpass', output='sos')

print(f"Áp dụng Elliptic riêng biệt:")
print(f"  - Amp: {LOWCUT_AMP}-{HIGHCUT_AMP} Hz (Order {ORDER_AMP})")
print(f"  - Phs: {LOWCUT_PHS}-{HIGHCUT_PHS} Hz (Order {ORDER_PHS})\n")

# ============ Áp dụng cho tất cả subcarriers ============
for sub_idx in ts_all.keys():
    ts_df = ts_all[sub_idx]
    
    # Input lấy từ bước trước (SG hoặc Hampel)
    amp_in = ts_df['amp_sg'].values if 'amp_sg' in ts_df.columns else ts_df['amp_hampel'].values
    phs_in = ts_df['phase_sg'].values if 'phase_sg' in ts_df.columns else ts_df['phase_hampel'].values
    
    # Tính toán padding
    pad_amp = min(PADLEN, len(amp_in) - 1)
    pad_phs = min(PADLEN, len(phs_in) - 1)
    
    # Lọc riêng biệt
    ts_all[sub_idx]['amp_ellip'] = sosfiltfilt(sos_amp, amp_in, padlen=pad_amp)
    ts_all[sub_idx]['phase_ellip'] = sosfiltfilt(sos_phs, phs_in, padlen=pad_phs)

print(f"Hoàn tất lọc Elliptic riêng biệt cho {len(ts_all)} subcarriers.")


Áp dụng Elliptic riêng biệt:
  - Amp: 0.15-0.5 Hz (Order 4)
  - Phs: 0.1-0.6 Hz (Order 2)

Hoàn tất lọc Elliptic riêng biệt cho 119 subcarriers.


## Display Sample

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Hiển thị Elliptic filter effect trên subcarrier 10
ts_df = ts_all[SUBCARRIER_INDEX]
packet_idx = np.arange(len(ts_df))

amp_before = ts_df['amp_sg'].values if 'amp_sg' in ts_df.columns else ts_df['amp_hampel'].values
phs_before = ts_df['phase_sg'].values if 'phase_sg' in ts_df.columns else ts_df['phase_hampel'].values

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        f"Amplitude Before Elliptic (SC {SUBCARRIER_INDEX})",
                        "Amplitude After Elliptic (0.1-0.4 Hz)",
                        f"Phase Before Elliptic (SC {SUBCARRIER_INDEX})",
                        "Phase After Elliptic (0.1-0.4 Hz)",
                    ])

fig.add_trace(go.Scatter(x=packet_idx, y=amp_before, mode='lines',
                          line=dict(color='royalblue', width=1), opacity=0.7,
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_ellip'], mode='lines',
                          line=dict(color='green', width=1), opacity=0.7,
                          showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=packet_idx, y=phs_before, mode='lines',
                          line=dict(color='darkorange', width=1), opacity=0.7,
                          showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase_ellip'], mode='lines',
                          line=dict(color='purple', width=1), opacity=0.7,
                          showlegend=False), row=2, col=2)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Phase (rad)", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=2)
fig.update_layout(height=600, width=1200, showlegend=False,
                  template='plotly_white')
fig.show()

In [13]:
import plotly.graph_objects as go
import numpy as np

# Plot tất cả subcarriers sau Elliptic filter trên cùng một biểu đồ
num_subcarriers = len(ts_all)

if num_subcarriers == 0:
    raise ValueError("ts_all rỗng, không có subcarrier để vẽ")

fig = go.Figure()

for sub_idx in sorted(ts_all.keys()):
    ts_df = ts_all[sub_idx]
    if 'amp_ellip' not in ts_df.columns:
        continue
    fig.add_trace(go.Scatter(
        x=np.arange(len(ts_df)),
        y=ts_df['amp_ellip'].values,
        mode='lines',
        line=dict(width=0.8),
        opacity=0.65,
        name=f'SC{sub_idx}',
        showlegend=False
    ))

fig.update_layout(
    title=f'All {num_subcarriers} Subcarriers After Elliptic Filter',
    xaxis_title='Packet Index',
    yaxis_title='Amplitude',
    height=600, width=1200,
    template='plotly_white'
)
fig.update_yaxes(range=[-10, 10])

fig.show()

print(f'Plotted {num_subcarriers} subcarriers after elliptic filter in single plot')

Plotted 119 subcarriers after elliptic filter in single plot


# PCA Dimensionality Reduction - Top 5 Components

In [14]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chuẩn bị ma trận dữ liệu: mỗi hàng là 1 packet, mỗi cột là 1 subcarrier
num_packets = max([len(ts_all[k]) for k in ts_all.keys()])
num_subcarriers = len(ts_all)

# Khởi tạo ma trận amplitude và phase
amp_matrix = np.zeros((num_packets, num_subcarriers))
phs_matrix = np.zeros((num_packets, num_subcarriers))

# Điền dữ liệu từ ts_all (sau khi đã qua filters: Hampel, SG, Elliptic)
for col_idx, sub_idx in enumerate(sorted(ts_all.keys())):
    ts_df = ts_all[sub_idx]
    n_samples = len(ts_df)
    amp_matrix[:n_samples, col_idx] = ts_df['amp_ellip'].values
    phs_matrix[:n_samples, col_idx] = ts_df['phase_ellip'].values

print(f"Ma trận amplitude: {amp_matrix.shape}")
print(f"Ma trận phase: {phs_matrix.shape}")

# Áp dụng PCA cho amplitude (giảm xuống 5 thành phần chính)
N_COMPONENTS = 5
pca_amp = PCA(n_components=N_COMPONENTS)
amp_pca = pca_amp.fit_transform(amp_matrix)

print(f"\nPCA Amplitude:")
print(f"  Input shape: {amp_matrix.shape}")
print(f"  Output shape: {amp_pca.shape}")
print(f"  Explained variance ratio: {pca_amp.explained_variance_ratio_}")
print(f"  Total variance explained: {np.sum(pca_amp.explained_variance_ratio_)*100:.2f}%")

# Áp dụng PCA cho phase
pca_phs = PCA(n_components=N_COMPONENTS)
phs_pca = pca_phs.fit_transform(phs_matrix)

print(f"\nPCA Phase:")
print(f"  Input shape: {phs_matrix.shape}")
print(f"  Output shape: {phs_pca.shape}")
print(f"  Explained variance ratio: {pca_phs.explained_variance_ratio_}")
print(f"  Total variance explained: {np.sum(pca_phs.explained_variance_ratio_)*100:.2f}%")

# Vẽ biểu đồ variance explained
pc_labels = [f'PC{i}' for i in range(1, N_COMPONENTS+1)]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['PCA Amplitude - Variance Explained',
                                    'PCA Phase - Variance Explained'])

# Amplitude variance
fig.add_trace(go.Bar(x=pc_labels, y=pca_amp.explained_variance_ratio_,
                     marker_color='royalblue', opacity=0.7, name='Amp Variance',
                     showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pc_labels, y=np.cumsum(pca_amp.explained_variance_ratio_),
                          mode='lines+markers', line=dict(color='red', width=2),
                          marker=dict(size=8), name='Cumulative'),
              row=1, col=1)

# Phase variance
fig.add_trace(go.Bar(x=pc_labels, y=pca_phs.explained_variance_ratio_,
                     marker_color='darkorange', opacity=0.7, name='Phs Variance',
                     showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=pc_labels, y=np.cumsum(pca_phs.explained_variance_ratio_),
                          mode='lines+markers', line=dict(color='red', width=2),
                          marker=dict(size=8), name='Cumulative',
                          showlegend=False),
              row=1, col=2)

fig.update_yaxes(title_text="Variance Explained Ratio", row=1, col=1)
fig.update_yaxes(title_text="Variance Explained Ratio", row=1, col=2)
fig.update_xaxes(title_text="Principal Component", row=1, col=1)
fig.update_xaxes(title_text="Principal Component", row=1, col=2)
fig.update_layout(height=500, width=1400, template='plotly_white')
fig.show()

print(f"\nPCA hoàn tất! Giảm từ {num_subcarriers} subcarriers xuống {N_COMPONENTS} principal components")

Ma trận amplitude: (4000, 119)
Ma trận phase: (4000, 119)

PCA Amplitude:
  Input shape: (4000, 119)
  Output shape: (4000, 5)
  Explained variance ratio: [0.73330553 0.17712094 0.02270532 0.01331449 0.01155858]
  Total variance explained: 95.80%

PCA Phase:
  Input shape: (4000, 119)
  Output shape: (4000, 5)
  Explained variance ratio: [0.62939133 0.13224031 0.08021598 0.0484787  0.02286568]
  Total variance explained: 91.32%



PCA hoàn tất! Giảm từ 119 subcarriers xuống 5 principal components


## Visualization - Top 5 Principal Components

In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Vẽ 10 thành phần chính
packet_idx = np.arange(len(amp_pca))

colors = ['royalblue', 'darkorange', 'green', 'red', 'purple',
          'brown', 'pink', 'gray', 'olive', 'cyan']

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=[
                        f'Amplitude - Top {N_COMPONENTS} Principal Components (Total: {np.sum(pca_amp.explained_variance_ratio_)*100:.1f}% variance)',
                        f'Phase - Top {N_COMPONENTS} Principal Components (Total: {np.sum(pca_phs.explained_variance_ratio_)*100:.1f}% variance)',
                    ])

# Plot Amplitude - skip PC1 (i starts from 1)
for i in range(0, N_COMPONENTS):
    fig.add_trace(go.Scatter(
        x=packet_idx, y=amp_pca[:, i], mode='lines',
        line=dict(color=colors[i], width=1.5), opacity=0.8,
        name=f'PC{i+1} ({pca_amp.explained_variance_ratio_[i]*100:.1f}%)',
        legendgroup='amp', legendgrouptitle_text='Amplitude'
    ), row=1, col=1)

# Plot Phase - all PCs
for i in range(0, 2):
    fig.add_trace(go.Scatter(
        x=packet_idx, y=phs_pca[:, i], mode='lines',
        line=dict(color=colors[i], width=1.5), opacity=0.8,
        name=f'PC{i+1} ({pca_phs.explained_variance_ratio_[i]*100:.1f}%)',
        legendgroup='phs', legendgrouptitle_text='Phase'
    ), row=2, col=1)

fig.update_yaxes(title_text="Amplitude (PCA)", row=1, col=1)
fig.update_yaxes(title_text="Phase (PCA)", row=2, col=1)
fig.update_yaxes(range=[-20, 20])
fig.update_xaxes(title_text="Packet Index", row=1, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_layout(height=600, width=1200, template='plotly_white')
fig.show()

print(f"\nPrincipal Components Ready:")
print(f"  amp_pca: shape {amp_pca.shape} - {N_COMPONENTS} principal components from amplitude")
print(f"  phs_pca: shape {phs_pca.shape} - {N_COMPONENTS} principal components from phase")


Principal Components Ready:
  amp_pca: shape (4000, 5) - 5 principal components from amplitude
  phs_pca: shape (4000, 5) - 5 principal components from phase


# Power Spectrogram (STFT) Feature Extraction for CNN

In [16]:
import numpy as np
from scipy.signal import spectrogram

# ============ Tham số Spectrogram ============
FS = 100.0          # Sampling frequency (Hz)
NPERSEG = 500       # Window length = 5s × 100 Hz
NOVERLAP = 450      # Overlap 90%
WINDOW = 'hann'     # Cửa sổ Hann

# Dải tần nhịp thở
BREATH_FREQ_MIN = 0.1   # Hz
BREATH_FREQ_MAX = 0.6   # Hz

print(f"Tính Power Spectrogram (STFT) cho {N_COMPONENTS} principal components...")
print(f"  Window: {WINDOW}, nperseg={NPERSEG} ({NPERSEG/FS:.1f}s)")
print(f"  Overlap: {NOVERLAP} ({NOVERLAP/NPERSEG*100:.0f}%)")
print(f"  Hop size: {NPERSEG - NOVERLAP} samples ({(NPERSEG - NOVERLAP)/FS:.2f}s)")
print(f"  Breathing band: [{BREATH_FREQ_MIN} - {BREATH_FREQ_MAX}] Hz\n")

def compute_spectrogram_matrix(pca_matrix, fs, nperseg, noverlap, window):
    """
    Tính spectrogram cho tất cả PCA components.
    Returns: freqs, times, list of Sxx matrices (one per component)
    """
    spectrograms = []
    freqs = None
    times = None
    
    for comp_idx in range(pca_matrix.shape[1]):
        f, t, Sxx = spectrogram(
            pca_matrix[:, comp_idx],
            fs=fs,
            window=window,
            nperseg=nperseg,
            noverlap=noverlap,
            scaling='density'   # Power Spectral Density (V²/Hz)
        )
        if freqs is None:
            freqs = f
            times = t
        spectrograms.append(Sxx)
    
    return freqs, times, spectrograms

def crop_freq_band(freqs, spectrograms, fmin, fmax):
    """Cắt dải tần quan tâm từ spectrogram."""
    mask = (freqs >= fmin) & (freqs <= fmax)
    cropped_freqs = freqs[mask]
    cropped_specs = [Sxx[mask, :] for Sxx in spectrograms]
    return cropped_freqs, cropped_specs

def to_decibel(spectrograms, ref=1e-12):
    """Chuyển Power Spectrogram sang dB: 10*log10(Sxx/ref)."""
    return [10 * np.log10(np.maximum(Sxx, ref) / ref) for Sxx in spectrograms]

# ============ Tính Spectrogram cho Amplitude PCA ============
spec_freqs, spec_times, spec_amp_full = compute_spectrogram_matrix(
    amp_pca, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP, window=WINDOW
)

# ============ Tính Spectrogram cho Phase PCA ============
_, _, spec_phs_full = compute_spectrogram_matrix(
    phs_pca, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP, window=WINDOW
)

print(f"Full Spectrogram:")
print(f"  Frequency bins: {len(spec_freqs)} ({spec_freqs[0]:.3f} - {spec_freqs[-1]:.3f} Hz)")
print(f"  Freq resolution: {spec_freqs[1] - spec_freqs[0]:.4f} Hz")
print(f"  Time frames: {len(spec_times)} ({spec_times[0]:.2f} - {spec_times[-1]:.2f} s)")
print(f"  Each component shape: {spec_amp_full[0].shape} (freq_bins × time_frames)")

# ============ Cắt dải tần nhịp thở ============
breath_freqs, breath_spec_amp = crop_freq_band(spec_freqs, spec_amp_full, BREATH_FREQ_MIN, BREATH_FREQ_MAX)
_, breath_spec_phs = crop_freq_band(spec_freqs, spec_phs_full, BREATH_FREQ_MIN, BREATH_FREQ_MAX)

print(f"\nBreathing Band [{BREATH_FREQ_MIN}-{BREATH_FREQ_MAX} Hz]:")
print(f"  Frequency bins: {len(breath_freqs)}")
print(f"  Each component shape: {breath_spec_amp[0].shape}")

# ============ Chuyển sang Decibel ============
breath_spec_amp_dB = to_decibel(breath_spec_amp)
breath_spec_phs_dB = to_decibel(breath_spec_phs)

print(f"\nDecibel range (Amplitude PC1):")
print(f"  Min: {np.min(breath_spec_amp_dB[0]):.1f} dB")
print(f"  Max: {np.max(breath_spec_amp_dB[0]):.1f} dB")

# ============ Stack thành tensor CNN-ready ============
# Shape: (N_COMPONENTS, n_freq_bins, n_time_frames) — giống multi-channel image
cnn_input_amp = np.stack(breath_spec_amp_dB, axis=0)   # (5, freq, time)
cnn_input_phs = np.stack(breath_spec_phs_dB, axis=0)   # (5, freq, time)

# Hoặc concat cả amp + phase: (10, freq, time)
cnn_input_combined = np.concatenate([cnn_input_amp, cnn_input_phs], axis=0)

print(f"\n{'='*60}")
print(f"CNN Input Tensors:")
print(f"  Amplitude only:  {cnn_input_amp.shape}  (channels=PC, freq, time)")
print(f"  Phase only:      {cnn_input_phs.shape}  (channels=PC, freq, time)")
print(f"  Combined:        {cnn_input_combined.shape}  (channels=amp+phs, freq, time)")
print(f"{'='*60}")

# Lưu feature matrices
feature_matrices = {
    'breath_freqs': breath_freqs,
    'spec_times': spec_times,
    'breath_spec_amp': breath_spec_amp,
    'breath_spec_phs': breath_spec_phs,
    'breath_spec_amp_dB': breath_spec_amp_dB,
    'breath_spec_phs_dB': breath_spec_phs_dB,
    'cnn_input_amp': cnn_input_amp,
    'cnn_input_phs': cnn_input_phs,
    'cnn_input_combined': cnn_input_combined,
}

print("\nFeature matrices saved!")

Tính Power Spectrogram (STFT) cho 5 principal components...
  Window: hann, nperseg=500 (5.0s)
  Overlap: 450 (90%)
  Hop size: 50 samples (0.50s)
  Breathing band: [0.1 - 0.6] Hz

Full Spectrogram:
  Frequency bins: 251 (0.000 - 50.000 Hz)
  Freq resolution: 0.2000 Hz
  Time frames: 71 (2.50 - 37.50 s)
  Each component shape: (251, 71) (freq_bins × time_frames)

Breathing Band [0.1-0.6 Hz]:
  Frequency bins: 2
  Each component shape: (2, 71)

Decibel range (Amplitude PC1):
  Min: 87.7 dB
  Max: 138.8 dB

CNN Input Tensors:
  Amplitude only:  (5, 2, 71)  (channels=PC, freq, time)
  Phase only:      (5, 2, 71)  (channels=PC, freq, time)
  Combined:        (10, 2, 71)  (channels=amp+phs, freq, time)

Feature matrices saved!


## Visualize Output

In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

colors_pc = ['royalblue', 'darkorange', 'green', 'red', 'purple',
             'brown', 'pink', 'gray', 'olive', 'cyan']

# ============================================================
# PLOT 1: Spectrogram heatmap cho từng PC (Amplitude + Phase, dB)
# ============================================================
fig1 = make_subplots(rows=N_COMPONENTS, cols=2,
                     subplot_titles=[t for i in range(N_COMPONENTS)
                                     for t in (f'Amplitude PC{i+1} (dB)',
                                               f'Phase PC{i+1} (dB)')],
                     vertical_spacing=0.03, horizontal_spacing=0.08)

for i in range(N_COMPONENTS):
    # Amplitude Spectrogram (dB)
    fig1.add_trace(go.Heatmap(
        z=breath_spec_amp_dB[i], x=spec_times, y=breath_freqs,
        colorscale='Viridis', colorbar=dict(title='dB', len=1/N_COMPONENTS, y=1 - (i+0.5)/N_COMPONENTS),
        showscale=(i == 0), name=f'Amp PC{i+1}'
    ), row=i+1, col=1)

    # Phase Spectrogram (dB)
    fig1.add_trace(go.Heatmap(
        z=breath_spec_phs_dB[i], x=spec_times, y=breath_freqs,
        colorscale='Magma', colorbar=dict(title='dB', len=1/N_COMPONENTS, y=1 - (i+0.5)/N_COMPONENTS, x=1.07),
        showscale=(i == 0), name=f'Phs PC{i+1}'
    ), row=i+1, col=2)

    fig1.update_yaxes(title_text='Freq (Hz)', row=i+1, col=1, title_font_size=9)

fig1.update_xaxes(title_text='Time (s)', row=N_COMPONENTS, col=1)
fig1.update_xaxes(title_text='Time (s)', row=N_COMPONENTS, col=2)
fig1.update_layout(height=350*N_COMPONENTS, width=1800, showlegend=False,
                   template='plotly_white',
                   title_text='Breathing Spectrograms per PC (dB)')
fig1.show()

# ============================================================
# PLOT 2: Tổng hợp — Mean Power theo thời gian & tần số
# ============================================================
fig2 = make_subplots(rows=2, cols=2,
                     subplot_titles=[
                         'Amplitude — Mean Breathing Power over Time',
                         'Phase — Mean Breathing Power over Time',
                         'Amplitude — Mean Breathing Spectrum',
                         'Phase — Mean Breathing Spectrum',
                     ])

for i in range(N_COMPONENTS):
    # Mean power over time (amplitude)
    mean_power_t = np.mean(breath_spec_amp_dB[i], axis=0)
    fig2.add_trace(go.Scatter(x=spec_times, y=mean_power_t, mode='lines',
                               line=dict(color=colors_pc[i], width=1.2), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='amp_t',
                               showlegend=True), row=1, col=1)

    # Mean power over time (phase)
    mean_power_t = np.mean(breath_spec_phs_dB[i], axis=0)
    fig2.add_trace(go.Scatter(x=spec_times, y=mean_power_t, mode='lines',
                               line=dict(color=colors_pc[i], width=1.2), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='phs_t',
                               showlegend=False), row=1, col=2)

    # Mean power over frequency (amplitude)
    mean_power_f = np.mean(breath_spec_amp_dB[i], axis=1)
    fig2.add_trace(go.Scatter(x=breath_freqs, y=mean_power_f, mode='lines',
                               line=dict(color=colors_pc[i], width=1.5), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='amp_f',
                               showlegend=False), row=2, col=1)

    # Mean power over frequency (phase)
    mean_power_f = np.mean(breath_spec_phs_dB[i], axis=1)
    fig2.add_trace(go.Scatter(x=breath_freqs, y=mean_power_f, mode='lines',
                               line=dict(color=colors_pc[i], width=1.5), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='phs_f',
                               showlegend=False), row=2, col=2)

fig2.update_xaxes(title_text='Time (s)', row=1, col=1)
fig2.update_xaxes(title_text='Time (s)', row=1, col=2)
fig2.update_xaxes(title_text='Frequency (Hz)', row=2, col=1)
fig2.update_xaxes(title_text='Frequency (Hz)', row=2, col=2)
fig2.update_yaxes(title_text='Mean Power (dB)', row=1, col=1)
fig2.update_yaxes(title_text='Mean Power (dB)', row=1, col=2)
fig2.update_yaxes(title_text='Mean Power (dB)', row=2, col=1)
fig2.update_yaxes(title_text='Mean Power (dB)', row=2, col=2)
fig2.update_layout(height=800, width=1600, template='plotly_white')
fig2.show()

# ============================================================
# PLOT 3: CNN Input Tensor preview (combined channels)
# ============================================================
fig3 = make_subplots(rows=2, cols=N_COMPONENTS,
                     subplot_titles=[f'Amp PC{i+1}' for i in range(N_COMPONENTS)]
                                    + [f'Phs PC{i+1}' for i in range(N_COMPONENTS)],
                     vertical_spacing=0.12, horizontal_spacing=0.04)

for i in range(N_COMPONENTS):
    # Amplitude channel
    fig3.add_trace(go.Heatmap(
        z=cnn_input_combined[i], x=spec_times, y=breath_freqs,
        colorscale='Viridis', showscale=False
    ), row=1, col=i+1)

    # Phase channel
    fig3.add_trace(go.Heatmap(
        z=cnn_input_combined[N_COMPONENTS + i], x=spec_times, y=breath_freqs,
        colorscale='Magma', showscale=False
    ), row=2, col=i+1)

    if i == 0:
        fig3.update_yaxes(title_text='Freq (Hz)', row=1, col=1)
        fig3.update_yaxes(title_text='Freq (Hz)', row=2, col=1)
    fig3.update_xaxes(title_text='Time (s)', row=2, col=i+1)

fig3.update_layout(
    height=500, width=350*N_COMPONENTS, showlegend=False,
    template='plotly_white',
    title_text=f'CNN Input Tensor — {cnn_input_combined.shape[0]} channels × '
               f'{cnn_input_combined.shape[1]} freq × {cnn_input_combined.shape[2]} time (dB)')
fig3.show()

# ============================================================
# Thống kê
# ============================================================
print(f"\n{'='*60}")
print("SPECTROGRAM STATISTICS (Breathing Band, dB)")
print(f"{'='*60}")
for i in range(N_COMPONENTS):
    print(f"\nPC{i+1}:")
    print(f"  Amplitude dB — min: {np.min(breath_spec_amp_dB[i]):.1f}, "
          f"max: {np.max(breath_spec_amp_dB[i]):.1f}, "
          f"mean: {np.mean(breath_spec_amp_dB[i]):.1f}")
    print(f"  Phase dB    — min: {np.min(breath_spec_phs_dB[i]):.1f}, "
          f"max: {np.max(breath_spec_phs_dB[i]):.1f}, "
          f"mean: {np.mean(breath_spec_phs_dB[i]):.1f}")

print(f"\n{'='*60}")
print(f"Spectrogram params: window={WINDOW} {NPERSEG/FS:.0f}s, "
      f"overlap={NOVERLAP/NPERSEG*100:.0f}%, hop={NPERSEG-NOVERLAP} samples")
print(f"CNN tensor shape: {cnn_input_combined.shape}")
print(f"{'='*60}")


SPECTROGRAM STATISTICS (Breathing Band, dB)

PC1:
  Amplitude dB — min: 87.7, max: 138.8, mean: 117.3
  Phase dB    — min: 105.0, max: 130.3, mean: 122.2

PC2:
  Amplitude dB — min: 91.2, max: 133.1, mean: 115.2
  Phase dB    — min: 89.2, max: 123.5, mean: 114.0

PC3:
  Amplitude dB — min: 79.9, max: 121.6, mean: 108.6
  Phase dB    — min: 89.4, max: 118.7, mean: 110.6

PC4:
  Amplitude dB — min: 89.6, max: 121.1, mean: 105.7
  Phase dB    — min: 83.9, max: 115.4, mean: 106.1

PC5:
  Amplitude dB — min: 86.4, max: 116.9, mean: 104.3
  Phase dB    — min: 90.7, max: 116.2, mean: 107.0

Spectrogram params: window=hann 5s, overlap=90%, hop=50 samples
CNN tensor shape: (10, 2, 71)


In [18]:
from scipy.fft import rfft, rfftfreq
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Breathing detection: Spectral Entropy -> FFT peak
# ============================================================

BREATH_FMIN = 0.10
BREATH_FMAX = 0.60
MAX_PCS_TO_CHECK = min(5, amp_pca.shape[1])

# Spectral entropy threshold (0.4 - 0.6)
SE_THRESHOLD = 0.50

print(f"filename = {latest_csv}")

# Compute spectral entropy in breathing band from STFT output
amp_spec = np.stack(breath_spec_amp, axis=0)  # (PC, freq, time)
amp_spec_mean = np.mean(amp_spec, axis=0)    # (freq, time)

Nf, Nt = amp_spec_mean.shape
entropies = np.zeros(Nt, dtype=float)
for t in range(Nt):
    p = amp_spec_mean[:, t].astype(float)
    total = p.sum()
    if total <= 0 or Nf <= 1:
        entropies[t] = 1.0
        continue
    q = p / total
    q_nonzero = q[q > 0]
    H = -np.sum(q_nonzero * np.log2(q_nonzero))
    entropies[t] = H / np.log2(Nf)

mean_SE = float(np.mean(entropies))
print("Spectral Entropy (normalized):", f"mean={mean_SE:.4f} (threshold={SE_THRESHOLD})")

if mean_SE >= SE_THRESHOLD:
    print("KET LUAN: phan tan roi loan -> khong co dong tac (no movement)")
    no_movement = True
else:
    print("KET LUAN: phan tap trung -> co dong tac, tiep tuc tim peak FFT")
    no_movement = False

n_samples = amp_pca.shape[0]

candidates = []
if not no_movement:
    for pc_idx in range(1, MAX_PCS_TO_CHECK):
        x = amp_pca[:, pc_idx].astype(float)
        x = x - np.mean(x)
        std_x = np.std(x)
        if std_x < 1e-12:
            continue
        x = x / std_x

        X = rfft(x)
        freqs = rfftfreq(n_samples, 1.0 / FS)
        power = np.abs(X) ** 2

        mask = (freqs >= BREATH_FMIN) & (freqs <= BREATH_FMAX)
        if not np.any(mask):
            continue

        freqs_band = freqs[mask]
        power_band = power[mask]

        peak_idx = int(np.argmax(power_band))
        freq_hz = float(freqs_band[peak_idx])
        bpm = freq_hz * 60.0
        peak_power = float(power_band[peak_idx])
        total_band_power = float(np.sum(power_band))
        rel_power = peak_power / total_band_power if total_band_power > 0 else 0.0

        candidates.append({
            'pc': pc_idx,
            'x': x,
            'freqs': freqs,
            'power': power,
            'freqs_band': freqs_band,
            'power_band': power_band,
            'freq_hz': freq_hz,
            'bpm': bpm,
            'peak_power': peak_power,
            'rel_power': rel_power,
        })

if no_movement:
    print("No movement, skipping FFT-based breathing frequency estimation.")
else:
    if len(candidates) == 0:
        print("=" * 60)
        print("Ket qua: KHONG TIM THAY DIEN TANH DOI CHO NHI PHAN THO.")
        print("Ket luan: KHONG CO DU DAU HIEU de uoc tinh nhip tho.")
        print("=" * 60)
    else:
        candidates = sorted(candidates, key=lambda c: (c['rel_power'], c['peak_power']), reverse=True)
        best = candidates[0]

        HARD_REL_POWER = 0.25
        SOFT_REL_POWER = 0.10

        if best['rel_power'] >= HARD_REL_POWER:
            state = 'RELIABLE'
        elif best['rel_power'] >= SOFT_REL_POWER:
            state = 'WEAK_CANDIDATE'
        else:
            state = 'UNRELIABLE'

        print("=" * 60)
        print(f"Dataset: {latest_csv}")
        print(f"PC duoc chon: PC{best['pc']+1}")
        print(f"Breath freq: {best['freq_hz']:.4f} Hz (~{best['bpm']:.1f} BPM)")
        print(f"Peak power (band): {best['peak_power']:.3f}, rel_power: {best['rel_power']:.3f}")
        print(f"Trang thai: {state}")
        if state == 'RELIABLE':
            print("Ket luan: CO TIN HIEU NHIP THO ro rang.")
        elif state == 'WEAK_CANDIDATE':
            print("Ket luan: CO DAU HIEU NHIP THO NHUNG CON YEU.")
        else:
            print("Ket luan: CHUA DU TIN CAY de ket luan co/khong co nhip tho.")
        print("=" * 60)

        x = best['x']
        freqs = best['freqs']
        power = best['power']
        freqs_band = best['freqs_band']
        power_band = best['power_band']

        fig = make_subplots(rows=2, cols=1,
                            subplot_titles=[
                                f"PC{best['pc']+1} Time Series (normalized)",
                                "FFT Power Spectrum within Respiration Band"
                            ],
                            vertical_spacing=0.12)

        fig.add_trace(go.Scatter(x=np.arange(len(x)) / FS, y=x,
                                 mode='lines', line=dict(color='royalblue', width=1.2),
                                 name='PC signal'), row=1, col=1)

        fig.add_trace(go.Scatter(x=freqs, y=power, mode='lines',
                                 line=dict(color='gray', width=1),
                                 name='FFT full'), row=2, col=1)

        fig.add_trace(go.Scatter(x=freqs_band, y=power_band, mode='lines',
                                 line=dict(color='firebrick', width=2),
                                 name='Respiration band'), row=2, col=1)

        fig.add_trace(go.Scatter(x=[best['freq_hz']], y=[best['peak_power']], mode='markers+text',
                                 text=[f"{best['bpm']:.1f} BPM"], textposition='top center',
                                 marker=dict(size=10, color='green'), showlegend=False), row=2, col=1)

        fig.update_xaxes(title_text='Time (s)', row=1, col=1)
        fig.update_xaxes(title_text='Frequency (Hz)', row=2, col=1)
        fig.update_yaxes(title_text='Normalized amplitude', row=1, col=1)
        fig.update_yaxes(title_text='Power', row=2, col=1)
        fig.update_layout(width=1100, height=700, template='plotly_white', showlegend=False)
        fig.show()

filename = Router/khong nguoi/csi_data_20260305_143656.csv
Spectral Entropy (normalized): mean=0.9423 (threshold=0.5)
KET LUAN: phan tan roi loan -> khong co dong tac (no movement)
No movement, skipping FFT-based breathing frequency estimation.


In [19]:
# # Quick classification helpers: use library routines for speed and consistency
# import time
# from pathlib import Path
# from csi_preprocessing.classifier import predict_from_csv_fast, predict_with_model_from_csv, predict_from_npz
# from csi_preprocessing.dataset import preprocess_csv_pipeline

# # Reuse the `latest_csv` variable defined earlier in the notebook.
# csv_path = Path(latest_csv)
# print(f'Starting quick checks for: {csv_path}')

# # 1) Fast heuristic (very fast, lightweight features)
# t0 = time.time()
# pred_fast, details_fast = predict_from_csv_fast(csv_path, n_packets_max=200)
# print(f'Fast heuristic -> pred={pred_fast} time={time.time()-t0:.3f}s')
# print(' details:', details_fast)

# # 2) Model-based prediction using cached model load (loads once per kernel)
# t0 = time.time()
# pred_model, details_model = predict_with_model_from_csv(csv_path, model_path='models/rf_person_detector_retrained.joblib')
# print(f'Model-based (light features) -> pred={pred_model} time={time.time()-t0:.3f}s')
# print(' details:', details_model)

# # 3) Full preprocess pipeline (more accurate but slower)
# t0 = time.time()
# out = preprocess_csv_pipeline(csv_path, Path('tmp_single_out'))
# t1 = time.time()
# print(f'Preprocess pipeline -> done in {t1-t0:.3f}s, output={out.get(output)}')
# pred_npz, details_npz = predict_from_npz(Path(out.get('output')) / 'dataset.npz')
# print(f'Predict from NPZ -> pred={pred_npz} time={time.time()-t0:.3f}s')
# print(' details:', details_npz)

# # Optional: cleanup temporary output (uncomment if desired)
# # import shutil
# # shutil.rmtree('tmp_single_out', ignore_errors=True)